# Multi-Task Preprocessing Pipeline

This notebook allows you to preprocess your TikTok dataset specifically for different NLP tasks:
1. **Sentiment Analysis**: Focuses on emotional valence.
2. **Intent Classification**: Focuses on functional triggers (appreciation, complaint, inquiry, recommendation, out of scope).
3. **Topic Classification**: Focuses on domain keywords (bouffe, price, treatment, service, endroit, delivery, unknown).

In [ ]:
import pandas as pd
import re
import emoji
import random

INPUT_FILE = 'dataset.csv'
SAMPLE_SIZE = 200 # Adjust as needed

## 1. Select Preprocessing Task
Run the cell below and type `1`, `2`, or `3` to set the preprocessing mode.

In [ ]:
print("Select Preprocessing Task:")
print("1. Sentiment Analysis")
print("2. Intent Classification")
print("3. Topic Classification")

choice = input("Enter choice (1/2/3): ")

modes = {"1": "SENTIMENT", "2": "INTENT", "3": "TOPIC"}
MODE = modes.get(choice, "SENTIMENT")
print(f"\n>>> Active Mode: {MODE}")

## 2. Core Preprocessing Functions

In [ ]:
def clean_structural(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"\[GIF\]|\[Sticker\]", "", text)
    text = re.sub(r"^(@[\w.]+[: ]*|Replying to @[\w.]+[: ]*)", "", text)
    text = re.sub(r"@\w+", "", text)
    return text.strip()

def map_emojis_by_task(text, mode):
    emojis_found = [e['emoji'] for e in emoji.emoji_list(text)]
    tokens = []
    
    if mode == "SENTIMENT":
        pos = ['❤️', '🥰', '😍', '🔥', '😋', '😂', '👏', '💯', '👍', '😁', '🤩', '😊', '🥳', '💪', '🤲', '🌹', '💐', '💎', '🇩🇿']
        neg = ['🤮', '😡', '👎', '💔', '💀', '💸', '😭', '😢', '😒', '😑', '😱']
        if any(e in pos for e in emojis_found): tokens.append("[POS]")
        if any(e in neg for e in emojis_found): tokens.append("[NEG]")
        
    elif mode == "INTENT":
        appr = ['❤️', '🥰', '😂', '👏', '🤲', '🌹', '💐']
        comp = ['🤮', '😡', '👎', '💔', '💀', '💸', '😒', '😑']
        inq = ['❓', '❔', '🤔', '🧐', '👀', '📍', '📞', '🕒']
        recom = ['👌', '🔝', '🌟', '✨', '✅', '🥇', '👑']
        
        if any(e in appr for e in emojis_found): tokens.append("[APPRECIATION]")
        if any(e in comp for e in emojis_found): tokens.append("[COMPLAINT]")
        if any(e in inq for e in emojis_found): tokens.append("[INQUIRY]")
        if any(e in recom for e in emojis_found): tokens.append("[RECOMMENDATION]")
        if not tokens and emojis_found: tokens.append("[OUT_OF_SCOPE]")
        
    elif mode == "TOPIC":
        bouffe = ['🥘', '🍔', '🍕', '🥙', '🥗', '🍦', '😋', '🤤', '🍜', '🍣', '🥩']
        price = ['💸', '💰', '💳', '💶', '💵']
        treat = ['🧑‍🍳', '👨‍🍳', '👋', '🤝', '🫂']
        srv = ['🕒', '⏳', '🛵', '🍴', '🍽️']
        endroit = ['📍', '🧼', '🧹', '📸', '🤳', '✨', '🌟', '🏝']
        delivery = ['🛵', '🚚', '📦']
        
        if any(e in bouffe for e in emojis_found): tokens.append("[BOUFFE]")
        if any(e in price for e in emojis_found): tokens.append("[PRICE]")
        if any(e in treat for e in emojis_found): tokens.append("[TREATMENT]")
        if any(e in srv for e in emojis_found): tokens.append("[SERVICE]")
        if any(e in endroit for e in emojis_found): tokens.append("[ENDROIT]")
        if any(e in delivery for e in emojis_found): tokens.append("[DELIVERY]")
        if not tokens and emojis_found: tokens.append("[UNKNOWN]")
        
    # Remove emojis and append tokens
    clean_text = emoji.replace_emoji(text, replace="")
    return (clean_text + " " + " ".join(tokens)).strip()

## 3. Execute Pipeline

In [ ]:
df = pd.read_csv(INPUT_FILE)
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Processing {len(sample_df)} rows for {MODE} task...")

sample_df['raw_text'] = sample_df['comment_text']
sample_df['comment_text'] = sample_df['comment_text'].apply(clean_structural)
sample_df['comment_text'] = sample_df['comment_text'].apply(lambda x: map_emojis_by_task(x, MODE))

# Final cleanup: dedup and shuffle
sample_df = sample_df.drop_duplicates(subset=['comment_text'])
sample_df = sample_df.sample(frac=1, random_state=42)
sample_df.insert(0, 'final_id', range(1, len(sample_df) + 1))

output_name = f'processed_sample_{MODE.lower()}.csv'
sample_df.to_csv(output_name, index=False)

print(f"Done! Saved to {output_name}")

In [ ]:
pd.set_option('display.max_colwidth', None)
display(sample_df[['raw_text', 'comment_text']].head(20))